# Max-P Regionalization

Authors: [Sergio Rey](https://github.com/sjsrey), [Xin Feng](https://github.com/xf37), [James Gaboardi](https://github.com/jGaboardi)

The `max-p` problem involves the clustering of a set of geographic areas into the maximum number of homogeneous regions such that the value of a spatially extensive regional attribute is above a predefined threshold value. The spatially extensive attribute can be specified to ensure that each region contains sufficient population size, or a minimum number of enumeration units. The number of regions $p$ is endogenous to the problem and is useful for regionalization problems where the analyst does not require a fixed number of regions a-priori.

Originally formulated as a mixed-integer problem in [Duque, Anselin, Rey (2012)](https://onlinelibrary.wiley.com/doi/abs/10.1111/j.1467-9787.2011.00743.x), `max-p` is an [NP-hard problem](https://en.wikipedia.org/wiki/NP-hardness) and exact solutions are only feasible for small problem sizes. As such, a number of heuristic solution approaches have been suggested. PySAL implements the heuristic approach described in
[Wei, Rey, and Knaap (2020)](https://www.tandfonline.com/doi/full/10.1080/13658816.2020.1759806).

In [ ]:
%config InlineBackend.figure_format = "retina"
%load_ext watermark
%watermark

In [ ]:
import warnings

import geopandas
import libpysal
import matplotlib.pyplot as plt
import numpy

import spopt
from spopt.region import MaxPHeuristic as MaxP

RANDOM_SEED = 123456

%watermark -w
%watermark -iv

## Mexican State Regional Income Clustering

To illustrate `maxp` we utilize data on regional incomes for Mexican states over the period 1940-2000, originally used in [Rey and Sastré-Gutiérrez (2010)](https://www.tandfonline.com/doi/full/10.1080/17421772.2010.493955?casa_token=r6iIZaEKaywAAAAA%3ARxjQkEOcxQ5gffa1EGyd2rByA6w0uVxg5cU3v4VaSmx2u89Nr3oSe2f_n63CV20Ysrjs07U47m98). 

We can first explore the data by plotting the per capital gross regional domestic product (in constant USD 2000 dollars) for each year in the sample, using a quintile classification. Here we will define a function for creating subplots useful in visual comparisons, which also can solve Max-P instances and will be used again later).

In [ ]:
def subplotter(gdf, incrs, w, threshold=5, top_n=2, seed=RANDOM_SEED):
    """Helper plotting function, also solves MaxP instances if desired."""
    rows, cols = incrs.shape
    f, axs = plt.subplots(rows, cols, figsize=(9, 14))
    for i in range(rows):
        for j in range(cols):
            year, _ax = incrs[i, j], axs[i, j]
            if not year:
                # plot country geographies
                _attr = "Mexico"
                gdf.plot(ax=_ax, ec="grey", fc="white")
            else:
                _attr = f"PCGDP{year}"
                if not w:
                    # plot country geographies by attributes
                    plt_kws = {"scheme": "Quantiles", "cmap": "GnBu", "ec": "grey"}
                    plt_kws.update({"legend_kwds": {"fmt": "{:.0f}"}})
                    gdf.plot(column=_attr, ax=_ax, legend=True, **plt_kws)
                else:
                    # solve a MaxP instance and plot regions
                    numpy.random.seed(seed)
                    maxp_args = gdf, w, _attr, "count", threshold
                    model = MaxP(*maxp_args, top_n=top_n)
                    model.solve()
                    label = year + "labels_"
                    gdf[label] = model.labels_
                    gdf.plot(column=label, ax=_ax, cmap="tab20")
            _ax.set_title(_attr)
            _ax.set_axis_off()
            _ax.set_aspect("equal")
    f.subplots_adjust(wspace=-0.7, hspace=-0.65)
    f.tight_layout()

In [ ]:
pth = libpysal.examples.get_path("mexicojoin.shp")
mexico = geopandas.read_file(pth)
mexico.head()

In [ ]:
mxgdp_years = [str(x) for x in range(1940, 2010, 10)]
mxgdp_years = numpy.array([None] + mxgdp_years)
mxgdp_years_grid = mxgdp_years.reshape(4, 2)
mxgdp_years_grid

### Regional incomes for Mexican states over the period 1940-2000

In [ ]:
subplotter(mexico, mxgdp_years_grid, None)

In general terms, the north-south divide in incomes is present in each of the 7 decades. There is some variation in states moving across quintiles however, and this is true at both the bottom and top of the state income distribution.

To develop a holistic view of the Mexican space economy over this timespan, we can try to form a set of spatially connected regions that maximizes the internal socieconomic levels of the states belonging to each region.

## Pooled Regionalization

To develop our holistic view, we can treat the six cross-sections as a multidimensional array and seek to 
cluster 32 Mexican states into the maximum number of regions such that each region as at least 6 = 32 // 5 states and homogeneity in per capita gross regional product over 1940-2000 is maximized.

We first define the variables in the dataframe that will be used to measure regional homogeneity:

In [ ]:
attrs_name = [f"PCGDP{year}" for year in mxgdp_years[1:]]
attrs_name

Next, we specify a number of parameters that will serve as input to the `max-p` model.

A [spatial weights object](https://pysal.org/libpysal/tutorial.html) expresses the spatial connectivity of the states:

In [ ]:
with warnings.catch_warnings(record=True):
    w = libpysal.weights.Queen.from_dataframe(mexico)

The remaining arguments are the minimum number of states each region must have (`threshold`):

In [ ]:
threshold = 6

and the number of the top candidate regions to consider when assigning enclaves (`top_n`):

In [ ]:
top_n = 2

We create the attribute `count` which will serve as the threshold attribute which we add to the dataframe:

In [ ]:
mexico["count"] = 1
threshold_name = "count"

The model can then be instantiated and solved:

In [ ]:
numpy.random.seed(RANDOM_SEED)
model = MaxP(mexico, w, attrs_name, threshold_name, threshold, top_n)
model.solve()

In [ ]:
mexico["maxp_new"] = model.labels_

In [ ]:
mexico[["maxp_new", "AREA"]].groupby(by="maxp_new").count()

In [ ]:
model.p

In [ ]:
mexico.plot(figsize=(8, 5), column="maxp_new", cmap="tab20", edgecolor="w").axis("off");

The model solution results in five regions, three of which have six states, and two with seven states each. Each region is a [spatially connected component](https://en.wikipedia.org/wiki/Connected_space), as required by the `max-p` problem.

## Change threshold to a minimum of 3 states per region

In [ ]:
numpy.random.seed(RANDOM_SEED)
threshold = 3
model = MaxP(mexico, w, attrs_name, threshold_name, threshold, top_n)
model.solve()

In [ ]:
mexico["maxp_3"] = model.labels_
mexico[["maxp_3", "AREA"]].groupby(by="maxp_3").count()

In [ ]:
model.p

In [ ]:
mexico.plot(figsize=(8, 5), column="maxp_3", cmap="tab20", edgecolor="w").axis("off");

## Year-by-Year Regionalization while Varying Parameters

### Vary minimum in-region threshold (5, 10, 15); hold enclave assignment constant (2)

**5 states**

In [ ]:
region_args = mexico, mxgdp_years_grid, w

In [ ]:
subplotter(*region_args, threshold=5, top_n=2)

**10 states**

In [ ]:
subplotter(*region_args, threshold=10, top_n=2)

**15 states**

In [ ]:
subplotter(*region_args, threshold=15, top_n=2)

###  Vary state enclave assignments (1, 3, 5); hold minimum in-region threshold constant (5)

**1 state**

In [ ]:
subplotter(*region_args, threshold=5, top_n=1)

**3 states**

In [ ]:
subplotter(*region_args, threshold=5, top_n=3)

**5 states**

In [ ]:
subplotter(*region_args, threshold=5, top_n=5)

As shown here, varying the `threshold` and `top_n` parameters both have a demonstrable effect regionalization outcomes.